### 교차 검증
- 데이터를 여러 부분으로 나눠서 여러 번 실험할으로써 **모델 품질을 더 정확하게 측정** 하는 방법

### 언제 교차 검증을 사용할까?
- 작은 데이터셋에서는 계산 시간이 크게 문제되지 않으므로, 교차 검증을 사용하는 것이 좋다.
- 큰 데이터셋에서는 단일 검증 세트로도 충분

In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import pandas as pd
from sklearn.model_selection import train_test_split

# 데이터 로드
melbourne_data = pd.read_csv('data/melb_data.csv')
y = melbourne_data.Price


melbourne_features = ['Type','Method','Regionname','Rooms','Distance','Postcode','Bedroom2', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude','Propertycount']
X = melbourne_data[melbourne_features]


# 훈련 데이터와 검증 데이터 분리
X_train, X_valid, y_train, y_valid = train_test_split(X, y,
                                                      train_size=0.8, test_size=0.2,
                                                      random_state=0)

# 숫자형 데이터 전처리
#- 누락된 값을 완성하기 위한 단변량 입력기
# - 각 열에 기술 통계 (ex. 평균, 중앙값 또는 가장 빈번한 값)를 사용하거나 상수 값을 사용하여 누락된 값을 바꾼다.
num_transformer = SimpleImputer(strategy='constant')

# 범주형 데이터 전처리 (결측값 대체 + 원-핫 인코딩)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

X_train.info()

# 범주형 데이터 뽑아내기
categorical_cols = [ col for col in X_train.columns if X_train[col].dtype=='object']
numberical_cols = [col for col in X_train.columns if X_train[col].dtype!='object']


# 숫자형 + 범주형 전처리 묶기
preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, numberical_cols),
        ('cat', categorical_transformer, categorical_cols)
    ]
)

from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(n_estimators=50, random_state=0)

my_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', model)
])

from sklearn.model_selection import cross_val_score

scores = -1 * cross_val_score(my_pipeline, X, y, cv=5, scoring='neg_mean_absolute_error')

print("MAE 점수들:\n", scores)

<class 'pandas.core.frame.DataFrame'>
Index: 14716 entries, 2573 to 2732
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Type           14716 non-null  object 
 1   Method         14716 non-null  object 
 2   Regionname     14715 non-null  object 
 3   Rooms          14716 non-null  int64  
 4   Distance       14715 non-null  float64
 5   Postcode       14715 non-null  float64
 6   Bedroom2       11937 non-null  float64
 7   Bathroom       11936 non-null  float64
 8   Landsize       10887 non-null  float64
 9   Lattitude      12041 non-null  float64
 10  Longtitude     12041 non-null  float64
 11  Propertycount  14715 non-null  float64
dtypes: float64(8), int64(1), object(3)
memory usage: 1.5+ MB
MAE 점수들:
 [227470.68297426 212781.02272794 198004.99943473 170449.75221934
 177215.31020231]


### 결론
- 교차 검증은 모델 품질을 훨씬 더 정확하게 측정할 수 있게 해준다.
- 파이프라인과 함께 사용하면 코드도 더 깔끔해지고, 별도의 검증 세트를 추적할 필요도 없다.